In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/uber-ride-analytics")
PROCESSED = PROJECT / "data" / "processed"

clean_path = PROCESSED / "uber_rides_cleaned.csv"

clean_df = pd.read_csv(clean_path)

print("Dataset shape:", clean_df.shape)
display(clean_df.head())

In [ ]:

total_bookings = len(clean_df)

status = (
    clean_df["booking_status"].astype("string").str.strip().str.lower()
    if "booking_status" in clean_df.columns
    else pd.Series("", index=clean_df.index)
)

completed = status.eq("completed").sum()

cancelled = status.str.contains(
    "cancel", na=False
).sum()

incomplete = status.str.contains(
    "incomplete", na=False
).sum()

kpis = {
    "Total bookings": total_bookings,
    "Completed bookings": int(completed),
    "Cancelled bookings": int(cancelled),
    "Incomplete bookings": int(incomplete),
    "Completion rate (%)": round(
        completed / total_bookings * 100, 2
    ) if total_bookings else 0,
    "Cancellation rate (%)": round(
        cancelled / total_bookings * 100, 2
    ) if total_bookings else 0
}

if "booking_value" in clean_df.columns:
    # Sum of reported booking values; validate the meaning of
    # this field before treating it as actual revenue.
    kpis["Total booking value"] = round(
        clean_df["booking_value"].sum(), 2
    )
    kpis["Average booking value"] = round(
        clean_df["booking_value"].mean(), 2
    )

if "ride_distance" in clean_df.columns:
    kpis["Average recorded ride distance"] = round(
        clean_df["ride_distance"].mean(), 2
    )

display(pd.DataFrame(
    list(kpis.items()), columns=["KPI", "Value"]
))


In [ ]:

sns.set_theme(style="whitegrid")

def save_chart(filename):
    plt.tight_layout()
    plt.savefig(VISUALS / filename, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

# Booking status
if "booking_status" in clean_df.columns:
    plt.figure(figsize=(10, 5))
    clean_df["booking_status"].value_counts().plot(
        kind="bar", color="steelblue"
    )
    plt.title("Booking Status Distribution")
    plt.xlabel("Booking Status")
    plt.ylabel("Number of Bookings")
    plt.xticks(rotation=35, ha="right")
    save_chart("booking_status.png")

# Hourly demand
if "booking_hour" in clean_df.columns:
    hourly = (
        clean_df["booking_hour"]
        .dropna().astype(int)
        .value_counts().reindex(range(24), fill_value=0)
    )

    plt.figure(figsize=(12, 5))
    plt.plot(hourly.index, hourly.values, marker="o")
    plt.xticks(range(24))
    plt.title("Bookings by Hour of Day")
    plt.xlabel("Hour (24-hour clock)")
    plt.ylabel("Number of Bookings")
    save_chart("hourly_demand.png")

# Vehicle preferences
if "vehicle_type" in clean_df.columns:
    plt.figure(figsize=(10, 5))
    clean_df["vehicle_type"].value_counts().plot(
        kind="bar", color="seagreen"
    )
    plt.title("Bookings by Vehicle Type")
    plt.xlabel("Vehicle Type")
    plt.ylabel("Bookings")
    plt.xticks(rotation=35, ha="right")
    save_chart("vehicle_types.png")

# Payment preferences
if "payment_method" in clean_df.columns:
    plt.figure(figsize=(9, 5))
    clean_df["payment_method"].value_counts().plot(
        kind="bar", color="darkorange"
    )
    plt.title("Bookings by Payment Method")
    plt.xlabel("Payment Method")
    plt.ylabel("Bookings")
    plt.xticks(rotation=35, ha="right")
    save_chart("payment_methods.png")

# Ride distance distribution
if "ride_distance" in clean_df.columns:
    distance = clean_df["ride_distance"].dropna()
    distance = distance[distance >= 0]

    plt.figure(figsize=(10, 5))
    plt.hist(distance, bins=40, edgecolor="black")
    plt.title("Distribution of Ride Distances")
    plt.xlabel("Ride Distance (dataset units)")
    plt.ylabel("Number of Records")
    save_chart("ride_distance.png")

print("Charts saved in:", VISUALS)


In [ ]:

if "booking_status" in clean_df.columns:
    cancellation_df = clean_df[
        clean_df["booking_status"].astype("string")
        .str.contains("cancel", case=False, na=False)
    ]

    print("Cancelled booking records:", len(cancellation_df))

    if "booking_hour" in cancellation_df.columns:
        cancelled_hourly = (
            cancellation_df["booking_hour"]
            .dropna().astype(int)
            .value_counts()
            .reindex(range(24), fill_value=0)
        )

        plt.figure(figsize=(12, 5))
        plt.bar(cancelled_hourly.index, cancelled_hourly.values)
        plt.xticks(range(24))
        plt.title("Cancelled Bookings by Hour")
        plt.xlabel("Hour of Day")
        plt.ylabel("Cancelled Bookings")
        save_chart("cancellations_by_hour.png")

    # Inspect cancellation reason columns
    reason_columns = [
        col for col in clean_df.columns
        if "reason" in col
    ]

    for col in reason_columns:
        print(f"\nTop values in {col}:")
        display(cancellation_df[col].value_counts().head(10))
